# INVENTORY.CSV

In [34]:
# ==========================================================
# INVENTORY.CSV AUDIT
# ==========================================================
import duckdb

con = duckdb.connect()
path = "../../source_database/seed_data/inventory.csv"

In [2]:
# 1. SCHEMA AUDIT
con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").show()# 2. COMPLETENESS AUDIT
total_rows = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchone()[0]
print(f"Total rows: {total_rows}")

cols = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()['column_name'].tolist()
for col in cols:
    null_count = con.sql(f"SELECT COUNT(*) - COUNT({col}) FROM read_csv_auto('{path}')").fetchone()[0]
    pct = round(null_count / total_rows * 100, 2)
    print(f"{col}: {null_count} nulls ({pct}%)")

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ inventory_id      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ store_id          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ product_id        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ quantity_on_hand  │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ quantity_reserved │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ quantity_on_order │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ reorder_level     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ last_counted_at   │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ is_adjustment     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ updated_at

In [9]:
# 2. COMPLETENESS AUDIT
total_rows = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchone()[0]
print(f"Total rows: {total_rows}")

cols = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()['column_name'].tolist()
for col in cols:
    null_count = con.sql(f"SELECT COUNT(*) - COUNT({col}) FROM read_csv_auto('{path}')").fetchone()[0]
    pct = round(null_count / total_rows * 100, 2)
    print(f"{col}: {null_count} nulls ({pct}%)")

Total rows: 385
inventory_id: 0 nulls (0.0%)
store_id: 0 nulls (0.0%)
product_id: 0 nulls (0.0%)
quantity_on_hand: 0 nulls (0.0%)
quantity_reserved: 0 nulls (0.0%)
quantity_on_order: 0 nulls (0.0%)
reorder_level: 0 nulls (0.0%)
last_counted_at: 0 nulls (0.0%)
is_adjustment: 0 nulls (0.0%)
updated_at: 0 nulls (0.0%)


In [3]:
# 3. UNIQUENESS AUDIT
# inventory_id is the primary key.
# Also check: should each (store_id, product_id) combination appear only ONCE?
# (a store shouldn't have two separate inventory rows for the same product)
dupe_pk = con.sql(f"""
    SELECT inventory_id, COUNT(*) AS row_count
    FROM read_csv_auto('{path}')
    GROUP BY inventory_id HAVING COUNT(*) > 1
""").df()
print(f"Duplicate inventory_id values: {len(dupe_pk)}")

dupe_combo = con.sql(f"""
    SELECT store_id, product_id, COUNT(*) AS row_count
    FROM read_csv_auto('{path}')
    GROUP BY store_id, product_id HAVING COUNT(*) > 1
""").df()
print(f"Duplicate (store_id, product_id) combinations: {len(dupe_combo)}")

Duplicate inventory_id values: 0
Duplicate (store_id, product_id) combinations: 0


In [4]:
# 4. REFERENTIAL INTEGRITY AUDIT
orphan_stores = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}') i
    LEFT JOIN read_csv_auto('../../source_database/seed_data/stores.csv') s ON i.store_id = s.store_id
    WHERE s.store_id IS NULL
""").fetchone()[0]
print(f"Orphaned store_id rows: {orphan_stores}")

orphan_products = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}') i
    LEFT JOIN read_csv_auto('../../source_database/seed_data/products.csv') p ON i.product_id = p.product_id
    WHERE p.product_id IS NULL
""").fetchone()[0]
print(f"Orphaned product_id rows: {orphan_products}")

Orphaned store_id rows: 0
Orphaned product_id rows: 0


In [12]:
# 5. VALIDITY / BUSINESS RULE AUDIT

# ==========================================================
# FIND NEGATIVE VALUES — checks ALL numeric columns automatically
# ==========================================================

# Step 1: get the list of numeric columns from the schema (so we don't have to
# hardcode column names, which is useful here since csv/xlsx use different ones)
schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
numeric_cols = schema[schema['column_type'].isin(['BIGINT', 'INTEGER', 'DOUBLE', 'DECIMAL'])]['column_name'].tolist()

print("Numeric columns found:", numeric_cols)
print()

# Step 2: check each one for negative values
for col in numeric_cols:
    neg_count = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}') WHERE {col} < 0").fetchone()[0]
    status = "⚠️ HAS NEGATIVES" if neg_count > 0 else "clean"
    print(f"{col}: {neg_count} negative values ({status})")


    
# quantity_on_hand should only be negative when is_adjustment = True
# (test the pattern, don't assume it — this confirms or breaks the theory)
bad_negative_qty = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE quantity_on_hand < 0 AND is_adjustment = FALSE
""").fetchone()[0]
print(f"Negative quantity_on_hand WITHOUT is_adjustment=True (should be 0): {bad_negative_qty}")

# quantities shouldn't be negative for reserved/on_order — these are always non-negative concepts
bad_reserved = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}') WHERE quantity_reserved < 0").fetchone()[0]
bad_on_order = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}') WHERE quantity_on_order < 0").fetchone()[0]
print(f"Negative quantity_reserved: {bad_reserved}")
print(f"Negative quantity_on_order: {bad_on_order}")

# reorder_level shouldn't be negative
bad_reorder = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}') WHERE reorder_level < 0").fetchone()[0]
print(f"Negative reorder_level: {bad_reorder}")

# last_counted_at shouldn't be in the future
future_count = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE CAST(last_counted_at AS TIMESTAMP) > CURRENT_TIMESTAMP
""").fetchone()[0]
print(f"last_counted_at in the future: {future_count}")


Numeric columns found: ['inventory_id', 'store_id', 'product_id', 'quantity_on_hand', 'quantity_reserved', 'quantity_on_order', 'reorder_level']

inventory_id: 0 negative values (clean)
store_id: 0 negative values (clean)
product_id: 0 negative values (clean)
quantity_on_hand: 30 negative values (⚠️ HAS NEGATIVES)
quantity_reserved: 0 negative values (clean)
quantity_on_order: 0 negative values (clean)
reorder_level: 0 negative values (clean)
Negative quantity_on_hand WITHOUT is_adjustment=True (should be 0): 0
Negative quantity_reserved: 0
Negative quantity_on_order: 0
Negative reorder_level: 0
last_counted_at in the future: 0


In [13]:
# 6. DISTRIBUTION / OUTLIER AUDIT
# Worth noting: quantity_on_hand is a DECIMAL, not a whole number (e.g. 171.6).
# That's unusual for a physical item count — flag it, don't just accept it.
con.sql(f"""
    SELECT
        MIN(quantity_on_hand) AS min_qty, MAX(quantity_on_hand) AS max_qty,
        AVG(quantity_on_hand) AS avg_qty
    FROM read_csv_auto('{path}')
""").show()

# how many values are non-whole numbers?
non_integer_qty = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE quantity_on_hand != ROUND(quantity_on_hand)
""").fetchone()[0]
print(f"quantity_on_hand values that aren't whole numbers: {non_integer_qty} out of {total_rows}")

┌─────────┬─────────┬────────────────────┐
│ min_qty │ max_qty │      avg_qty       │
│ double  │ double  │       double       │
├─────────┼─────────┼────────────────────┤
│   -49.4 │   499.2 │ 220.23324675324685 │
└─────────┴─────────┴────────────────────┘

quantity_on_hand values that aren't whole numbers: 338 out of 385


In [7]:
# 7. FRESHNESS / VOLUME AUDIT
con.sql(f"""
    SELECT MIN(CAST(last_counted_at AS TIMESTAMP)) AS earliest_count,
           MAX(CAST(last_counted_at AS TIMESTAMP)) AS latest_count,
           COUNT(*) AS total_rows
    FROM read_csv_auto('{path}')
""").show()

┌─────────────────────┬─────────────────────┬────────────┐
│   earliest_count    │    latest_count     │ total_rows │
│      timestamp      │      timestamp      │   int64    │
├─────────────────────┼─────────────────────┼────────────┤
│ 2026-01-01 09:36:00 │ 2026-06-30 15:53:00 │        385 │
└─────────────────────┴─────────────────────┴────────────┘



In [8]:
# 8. FORMAT CONSISTENCY AUDIT
con.sql(f"SELECT DISTINCT is_adjustment FROM read_csv_auto('{path}')").show()

┌───────────────┐
│ is_adjustment │
│    boolean    │
├───────────────┤
│ false         │
│ true          │
└───────────────┘



In [35]:
# 9. Distribution anomaly check
for col in ['store_id']:
    print(f"\n-- {col} --")
    print(con.sql(f"""
        SELECT {col}, COUNT(*) AS n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 1) AS pct
        FROM read_csv_auto('{path}') GROUP BY {col} ORDER BY n DESC
    """).df())


-- store_id --
    store_id   n  pct
0          7  20  5.2
1          4  20  5.2
2          9  17  4.4
3          5  17  4.4
4         10  17  4.4
5         25  17  4.4
6         12  17  4.4
7         17  16  4.2
8         24  16  4.2
9         28  14  3.6
10        29  13  3.4
11        11  12  3.1
12        22  12  3.1
13        16  12  3.1
14        18  12  3.1
15         1  12  3.1
16        23  12  3.1
17        15  11  2.9
18        30  11  2.9
19         2  11  2.9
20        13  11  2.9
21         6  11  2.9
22        14  10  2.6
23         8  10  2.6
24        27  10  2.6
25        21  10  2.6
26         3  10  2.6
27        26   9  2.3
28        20   8  2.1
29        19   7  1.8


In [36]:
# --- 10. Timezone drift: hour-of-day distribution ---
# last_counted_at
print("last_counted_at")
print(con.sql(f"""
    SELECT EXTRACT(hour FROM last_counted_at) AS hour, COUNT(*) AS n
    FROM read_csv_auto('{path}') GROUP BY hour ORDER BY hour
""").df())

# updated_at
print("\nupdated_at")
print(con.sql(f"""
    SELECT EXTRACT(hour FROM updated_at) AS hour, COUNT(*) AS n
    FROM read_csv_auto('{path}') GROUP BY hour ORDER BY hour
""").df())

last_counted_at
    hour   n
0      6  33
1      7  25
2      8  32
3      9  26
4     10  15
5     11  19
6     12  21
7     13  16
8     14  19
9     15  18
10    16  22
11    17  22
12    18  21
13    19  34
14    20  15
15    21  18
16    22  29

updated_at
   hour    n
0     0  385


In [38]:
results = []

# last_counted_at
# 1. Hour-of-day spread (timezone drift check)
hours = con.sql(f"SELECT EXTRACT(hour FROM last_counted_at) AS hour, COUNT(*) AS n FROM read_csv_auto('{path}') GROUP BY hour").df()
results.append({
    'check': 'last_counted_at hour-of-day coverage',
    'result': f"{hours['hour'].nunique()} distinct hours used (range {int(hours['hour'].min())}-{int(hours['hour'].max())})",
    'status': 'OK'
})

# 2. Is updated_at a real timestamp or a constant placeholder?
stats = con.sql(f"""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT updated_at) AS distinct_dates
    FROM read_csv_auto('{path}')
""").df().iloc[0]

results.append({
    'check': 'updated_at distinct values',
    'result': f"{stats['distinct_dates']} distinct value(s) across {stats['total_rows']} rows",
    'status': 'OK' if stats['distinct_dates'] > 1 else '⚠️ FLAG — placeholder, not real'
})

# 3. Gaps in inventory_id sequence
gaps = con.sql(f"""
    WITH bounds AS (SELECT MIN(inventory_id) mn, MAX(inventory_id) mx FROM read_csv_auto('{path}')),
    spine AS (SELECT UNNEST(generate_series(mn, mx)) AS id FROM bounds)
    SELECT COUNT(*) AS n FROM spine s LEFT JOIN read_csv_auto('{path}') c ON s.id = c.inventory_id WHERE c.inventory_id IS NULL
""").fetchone()[0]
results.append({
    'check': 'inventory_id sequence gaps',
    'result': f"{gaps} missing ID(s)",
    'status': 'OK' if gaps == 0 else '⚠️ FLAG'
})

summary = pd.DataFrame(results)
summary

,check,result,status
0,last_counted_at hour-of-day coverage,17 distinct hours used (range 6-22),OK
1,updated_at distinct values,1 distinct value(s) across 385 rows,"⚠️ FLAG — placeholder, not real"
2,inventory_id sequence gaps,0 missing ID(s),OK


In [39]:
# 11. Format consistency: Flag columns that contain mixed data types
def format_consistency_check(con, path):
    # Step 1: get every column name in the file
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()

    all_rows = []
    for col in columns:
        # Step 2: for THIS column, classify every value into a bucket
        breakdown = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                CASE
                    WHEN "{col}" IS NULL OR TRIM("{col}") = '' THEN 'NULL/EMPTY'
                    WHEN TRY_CAST("{col}" AS BIGINT) IS NOT NULL THEN 'INTEGER'
                    WHEN TRY_CAST("{col}" AS DOUBLE) IS NOT NULL THEN 'DECIMAL'
                    WHEN TRY_CAST("{col}" AS DATE) IS NOT NULL THEN 'DATE'
                    WHEN TRY_CAST("{col}" AS TIMESTAMP) IS NOT NULL THEN 'TIMESTAMP'
                    WHEN lower("{col}") IN ('true','false') THEN 'BOOLEAN'
                    ELSE 'TEXT'
                END AS detected_type,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            GROUP BY detected_type
            ORDER BY row_count DESC
        """).df()
        all_rows.append(breakdown)

    result = pd.concat(all_rows, ignore_index=True)

    # Step 3: flag a column only if it has 2+ real (non-null) types
    n_types = result[result['detected_type'] != 'NULL/EMPTY'].groupby('column_name')['detected_type'].transform('nunique')
    result['is_mixed'] = (n_types.reindex(result.index).fillna(0) > 1)
    return result

report = format_consistency_check(con, path)
report                       # full breakdown — bare, no print(), so it renders as a table

,column_name,detected_type,row_count,is_mixed
0,inventory_id,INTEGER,385,False
1,store_id,INTEGER,385,False
2,product_id,INTEGER,385,False
3,quantity_on_hand,INTEGER,385,False
4,quantity_reserved,INTEGER,385,False
5,quantity_on_order,INTEGER,385,False
6,reorder_level,INTEGER,385,False
7,last_counted_at,DATE,385,False
8,is_adjustment,BOOLEAN,385,False
9,updated_at,DATE,385,False


In [40]:
# 12. Sentinel/Default value usage:

# Part 1: scan for known/common sentinel patterns
SENTINEL_VALUES = [
    '-1', '0', '999', '9999', '99999', '-999',
    'NA', 'N/A', 'NULL', 'NONE', 'UNKNOWN', 'UNK', 'TBD', '-', '?',
    '1900-01-01', '1970-01-01', '9999-12-31', '0001-01-01'
]

def sentinel_scan(con, path, sentinel_values):
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()
    values_sql = ", ".join(f"'{v.upper()}'" for v in sentinel_values)

    all_rows = []
    for col in columns:
        df = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                "{col}" AS value,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            WHERE upper(TRIM("{col}")) IN ({values_sql})
            GROUP BY "{col}"
            ORDER BY row_count DESC
        """).df()
        all_rows.append(df)

    return pd.concat(all_rows, ignore_index=True)

sentinel_report = sentinel_scan(con, path, SENTINEL_VALUES)
sentinel_report


,column_name,value,row_count


# INVENTORY.XLSX

In [14]:
import duckdb
import pandas as pd

xlsx_path = "../../source_database/seed_data/inventory.xlsx"
inventory_xlsx = pd.read_excel(xlsx_path)

# side-step DuckDB's register()/read_xlsx issues — export to CSV, then read that
inventory_xlsx.to_csv("inventory_xlsx_export.csv", index=False)

con = duckdb.connect()
path = "inventory_xlsx_export.csv"   # everything below uses this variable name

In [15]:
# 1. SCHEMA AUDIT — columns and inferred types
con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").show()

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ inventory_id      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ store_id          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ product_id        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ quantity_on_hand  │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ quantity_reserved │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ quantity_on_order │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ reorder_point     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ last_updated      │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ is_adjustment     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ expiry_dat

In [16]:
# 2. COMPLETENESS AUDIT
total_rows = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchone()[0]
print(f"Total rows: {total_rows}")

cols = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()['column_name'].tolist()
for col in cols:
    null_count = con.sql(f"SELECT COUNT(*) - COUNT({col}) FROM read_csv_auto('{path}')").fetchone()[0]
    pct = round(null_count / total_rows * 100, 2)
    print(f"{col}: {null_count} nulls ({pct}%)")

Total rows: 385
inventory_id: 0 nulls (0.0%)
store_id: 0 nulls (0.0%)
product_id: 0 nulls (0.0%)
quantity_on_hand: 0 nulls (0.0%)
quantity_reserved: 0 nulls (0.0%)
quantity_on_order: 0 nulls (0.0%)
reorder_point: 0 nulls (0.0%)
last_updated: 0 nulls (0.0%)
is_adjustment: 0 nulls (0.0%)
expiry_date: 0 nulls (0.0%)


In [17]:
# 3. UNIQUENESS AUDIT
# inventory_id is the primary key.
# Also check: should each (store_id, product_id) combination appear only ONCE?
# (a store shouldn't have two separate inventory rows for the same product)
dupe_pk = con.sql(f"""
    SELECT inventory_id, COUNT(*) AS row_count
    FROM read_csv_auto('{path}')
    GROUP BY inventory_id HAVING COUNT(*) > 1
""").df()
print(f"Duplicate inventory_id values: {len(dupe_pk)}")

dupe_combo = con.sql(f"""
    SELECT store_id, product_id, COUNT(*) AS row_count
    FROM read_csv_auto('{path}')
    GROUP BY store_id, product_id HAVING COUNT(*) > 1
""").df()
print(f"Duplicate (store_id, product_id) combinations: {len(dupe_combo)}")

Duplicate inventory_id values: 0
Duplicate (store_id, product_id) combinations: 0


In [18]:
# 4. REFERENTIAL INTEGRITY AUDIT
orphan_stores = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}') i
    LEFT JOIN read_csv_auto('../../source_database/seed_data/stores.csv') s ON i.store_id = s.store_id
    WHERE s.store_id IS NULL
""").fetchone()[0]
print(f"Orphaned store_id rows: {orphan_stores}")

orphan_products = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}') i
    LEFT JOIN read_csv_auto('../../source_database/seed_data/products.csv') p ON i.product_id = p.product_id
    WHERE p.product_id IS NULL
""").fetchone()[0]
print(f"Orphaned product_id rows: {orphan_products}")

Orphaned store_id rows: 0
Orphaned product_id rows: 0


In [21]:
# 5. VALIDITY / BUSINESS RULE AUDIT

# ==========================================================
# FIND NEGATIVE VALUES — checks ALL numeric columns automatically
# ==========================================================

# Step 1: get the list of numeric columns from the schema (so we don't have to
# hardcode column names, which is useful here since csv/xlsx use different ones)
schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
numeric_cols = schema[schema['column_type'].isin(['BIGINT', 'INTEGER', 'DOUBLE', 'DECIMAL'])]['column_name'].tolist()

print("Numeric columns found:", numeric_cols)
print()

# Step 2: check each one for negative values
for col in numeric_cols:
    neg_count = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}') WHERE {col} < 0").fetchone()[0]
    status = "⚠️ HAS NEGATIVES" if neg_count > 0 else "clean"
    print(f"{col}: {neg_count} negative values ({status})")


    
# quantity_on_hand should only be negative when is_adjustment = True
# (test the pattern, don't assume it — this confirms or breaks the theory)
bad_negative_qty = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE quantity_on_hand < 0 AND is_adjustment = FALSE
""").fetchone()[0]
print(f"Negative quantity_on_hand WITHOUT is_adjustment=True (should be 0): {bad_negative_qty}")

# quantities shouldn't be negative for reserved/on_order — these are always non-negative concepts
bad_reserved = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}') WHERE quantity_reserved < 0").fetchone()[0]
bad_on_order = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}') WHERE quantity_on_order < 0").fetchone()[0]
print(f"Negative quantity_reserved: {bad_reserved}")
print(f"Negative quantity_on_order: {bad_on_order}")

# reorder_point shouldn't be negative
bad_reorder = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}') WHERE reorder_point < 0").fetchone()[0]
print(f"Negative reorder_level: {bad_reorder}")

# last_updated shouldn't be in the future
future_count = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE CAST(last_updated AS TIMESTAMP) > CURRENT_TIMESTAMP
""").fetchone()[0]
print(f"last_counted_at in the future: {future_count}")


Numeric columns found: ['inventory_id', 'store_id', 'product_id', 'quantity_on_hand', 'quantity_reserved', 'quantity_on_order', 'reorder_point']

inventory_id: 0 negative values (clean)
store_id: 0 negative values (clean)
product_id: 0 negative values (clean)
quantity_on_hand: 11 negative values (⚠️ HAS NEGATIVES)
quantity_reserved: 0 negative values (clean)
quantity_on_order: 0 negative values (clean)
reorder_point: 0 negative values (clean)
Negative quantity_on_hand WITHOUT is_adjustment=True (should be 0): 0
Negative quantity_reserved: 0
Negative quantity_on_order: 0
Negative reorder_level: 0
last_counted_at in the future: 0


In [22]:
# 6. DISTRIBUTION / OUTLIER AUDIT
# Worth noting: quantity_on_hand is a DECIMAL, not a whole number (e.g. 171.6).
# That's unusual for a physical item count — flag it, don't just accept it.
con.sql(f"""
    SELECT
        MIN(quantity_on_hand) AS min_qty, MAX(quantity_on_hand) AS max_qty,
        AVG(quantity_on_hand) AS avg_qty
    FROM read_csv_auto('{path}')
""").show()

# how many values are non-whole numbers?
non_integer_qty = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE quantity_on_hand != ROUND(quantity_on_hand)
""").fetchone()[0]
print(f"quantity_on_hand values that aren't whole numbers: {non_integer_qty} out of {total_rows}")

┌─────────┬─────────┬────────────────────┐
│ min_qty │ max_qty │      avg_qty       │
│ double  │ double  │       double       │
├─────────┼─────────┼────────────────────┤
│   -18.7 │   299.9 │ 144.49272727272722 │
└─────────┴─────────┴────────────────────┘

quantity_on_hand values that aren't whole numbers: 344 out of 385


In [24]:
# 7. FRESHNESS / VOLUME AUDIT
con.sql(f"""
    SELECT MIN(CAST(last_updated AS TIMESTAMP)) AS earliest_count,
           MAX(CAST(last_updated AS TIMESTAMP)) AS latest_count,
           COUNT(*) AS total_rows
    FROM read_csv_auto('{path}')
""").show()

┌─────────────────────┬─────────────────────┬────────────┐
│   earliest_count    │    latest_count     │ total_rows │
│      timestamp      │      timestamp      │   int64    │
├─────────────────────┼─────────────────────┼────────────┤
│ 2026-01-01 00:00:00 │ 2026-07-28 00:00:00 │        385 │
└─────────────────────┴─────────────────────┴────────────┘



In [25]:
# 8. FORMAT CONSISTENCY AUDIT
con.sql(f"SELECT DISTINCT is_adjustment FROM read_csv_auto('{path}')").show()

┌───────────────┐
│ is_adjustment │
│    boolean    │
├───────────────┤
│ false         │
│ true          │
└───────────────┘



# Comparision between csv and xlsx

In [26]:
# ==========================================================
# INVENTORY.CSV vs INVENTORY.XLSX — FULL COMPARISON AUDIT
# ==========================================================
import pandas as pd

csv_path = "../../source_database/seed_data/inventory.csv"
xlsx_path = "../../source_database/seed_data/inventory.xlsx"

csv_df = pd.read_csv(csv_path)
xlsx_df = pd.read_excel(xlsx_path)

print(f"inventory.csv  -> {csv_df.shape[0]} rows, {csv_df.shape[1]} columns")
print(f"inventory.xlsx -> {xlsx_df.shape[0]} rows, {xlsx_df.shape[1]} columns")

inventory.csv  -> 385 rows, 10 columns
inventory.xlsx -> 385 rows, 10 columns


In [27]:
# STEP 1 — SCHEMA COMPARISON
# We already know 3 columns are renamed, not missing — confirm that here rather than assume it.
csv_cols = set(csv_df.columns)
xlsx_cols = set(xlsx_df.columns)

only_in_csv  = csv_cols - xlsx_cols
only_in_xlsx = xlsx_cols - csv_cols
shared_cols  = csv_cols & xlsx_cols

print("Columns only in csv :", sorted(only_in_csv))
print("Columns only in xlsx:", sorted(only_in_xlsx))
print("Columns in both (same name)  :", sorted(shared_cols))

# Explicit rename map — reorder_level/reorder_point and last_counted_at/last_updated
# are almost certainly the SAME concept under a different name.
# expiry_date has no csv equivalent at all — it's a genuinely new field, not a rename.
rename_map = {"reorder_level": "reorder_point", "last_counted_at": "last_updated"}
print("\nAssumed renamed-but-same fields:", rename_map)
print("Field with no csv equivalent: updated_at (csv) / expiry_date (xlsx) — NOT treated as a rename")

Columns only in csv : ['last_counted_at', 'reorder_level', 'updated_at']
Columns only in xlsx: ['expiry_date', 'last_updated', 'reorder_point']
Columns in both (same name)  : ['inventory_id', 'is_adjustment', 'product_id', 'quantity_on_hand', 'quantity_on_order', 'quantity_reserved', 'store_id']

Assumed renamed-but-same fields: {'reorder_level': 'reorder_point', 'last_counted_at': 'last_updated'}
Field with no csv equivalent: updated_at (csv) / expiry_date (xlsx) — NOT treated as a rename


In [28]:
# STEP 2 — inventory_id OVERLAP
csv_ids  = set(csv_df["inventory_id"])
xlsx_ids = set(xlsx_df["inventory_id"])

only_in_csv_ids  = csv_ids - xlsx_ids
only_in_xlsx_ids = xlsx_ids - csv_ids
shared_ids       = csv_ids & xlsx_ids

print(f"inventory_id only in csv : {len(only_in_csv_ids)}")
print(f"inventory_id only in xlsx: {len(only_in_xlsx_ids)}")
print(f"inventory_id in both     : {len(shared_ids)}")

inventory_id only in csv : 0
inventory_id only in xlsx: 0
inventory_id in both     : 385


In [29]:
# STEP 3 — DOES inventory_id EVEN MEAN THE SAME THING IN BOTH FILES?
# inventory_id is a surrogate key — it's only trustworthy if it points to the
# SAME (store_id, product_id) pair in both files. Check that before trusting any
# further comparison that joins on inventory_id.
key_check = csv_df.merge(xlsx_df, on="inventory_id", suffixes=("_csv", "_xlsx"))
mismatched_keys = key_check[
    (key_check["store_id_csv"] != key_check["store_id_xlsx"]) |
    (key_check["product_id_csv"] != key_check["product_id_xlsx"])
]
print(f"inventory_id rows where (store_id, product_id) DON'T match between files: {len(mismatched_keys)}")
print("If this is > 0, inventory_id is NOT a safe join key between these two files.")

inventory_id rows where (store_id, product_id) DON'T match between files: 385
If this is > 0, inventory_id is NOT a safe join key between these two files.


In [30]:
# STEP 4 — FIELD-BY-FIELD COMPARISON FOR SHARED-NAME COLUMNS
compare_cols = sorted(shared_cols - {"inventory_id"})

merged = csv_df.merge(xlsx_df, on="inventory_id", suffixes=("_csv", "_xlsx"))
print(f"Rows compared (inventory_id present in both files): {len(merged)}")

mismatch_summary = {}
for col in compare_cols:
    mismatches = merged[f"{col}_csv"] != merged[f"{col}_xlsx"]
    mismatch_summary[col] = {
        "mismatches": int(mismatches.sum()),
        "pct": round(mismatches.sum() / len(merged) * 100, 2)
    }

mismatch_df = pd.DataFrame(mismatch_summary).T.sort_values("pct", ascending=False)
mismatch_df

Rows compared (inventory_id present in both files): 385


,mismatches,pct
quantity_on_hand,385.0,100.00
quantity_on_order,385.0,100.00
quantity_reserved,385.0,100.00
product_id,382.0,99.22
store_id,371.0,96.36
is_adjustment,41.0,10.65


In [31]:
# STEP 5 — COMPARE THE RENAMED FIELDS SEPARATELY (reorder_level vs reorder_point, etc.)
for csv_col, xlsx_col in rename_map.items():
    mismatches = merged[csv_col] != merged[xlsx_col]
    pct = round(mismatches.sum() / len(merged) * 100, 2)
    print(f"{csv_col} (csv) vs {xlsx_col} (xlsx): {mismatches.sum()} mismatches ({pct}%)")

reorder_level (csv) vs reorder_point (xlsx): 383 mismatches (99.48%)
last_counted_at (csv) vs last_updated (xlsx): 385 mismatches (100.0%)


In [32]:
# STEP 6 — SAMPLE EVIDENCE ROWS
worst_col = mismatch_df.index[0]
sample_mismatch = merged[merged[f"{worst_col}_csv"] != merged[f"{worst_col}_xlsx"]][
    ["inventory_id", "store_id_csv", "product_id_csv", f"{worst_col}_csv", f"{worst_col}_xlsx"]
].head(10)
print(f"Sample mismatches on: {worst_col}")
sample_mismatch

Sample mismatches on: quantity_on_hand


,inventory_id,store_id_csv,product_id_csv,quantity_on_hand_csv,quantity_on_hand_xlsx
0,1,9,120,171.6,59.4
1,2,23,134,127.5,137.0
2,3,8,131,306.6,160.3
3,4,20,131,185.8,270.2
4,5,26,81,329.5,144.0
5,6,25,120,361.2,230.6
6,7,11,61,495.3,155.8
7,8,24,147,282.9,284.1
8,9,28,97,351.8,58.4
9,10,5,128,-43.9,74.1


In [33]:
# STEP 7 — FINAL SUMMARY
print("=== inventory.csv vs inventory.xlsx — comparison summary ===")
print(f"inventory.csv : {len(csv_df)} rows, {len(csv_df.columns)} columns")
print(f"inventory.xlsx: {len(xlsx_df)} rows, {len(xlsx_df.columns)} columns")
print(f"Columns only in csv : {sorted(only_in_csv)}")
print(f"Columns only in xlsx: {sorted(only_in_xlsx)}")
print(f"inventory_id shared between files: {len(shared_ids)}")
print(f"inventory_id rows with mismatched (store_id, product_id): {len(mismatched_keys)}")
print()
print("Field mismatch rate on shared-name columns:")
print(mismatch_df)

=== inventory.csv vs inventory.xlsx — comparison summary ===
inventory.csv : 385 rows, 10 columns
inventory.xlsx: 385 rows, 10 columns
Columns only in csv : ['last_counted_at', 'reorder_level', 'updated_at']
Columns only in xlsx: ['expiry_date', 'last_updated', 'reorder_point']
inventory_id shared between files: 385
inventory_id rows with mismatched (store_id, product_id): 385

Field mismatch rate on shared-name columns:
                   mismatches     pct
quantity_on_hand        385.0  100.00
quantity_on_order       385.0  100.00
quantity_reserved       385.0  100.00
product_id              382.0   99.22
store_id                371.0   96.36
is_adjustment            41.0   10.65
